# 04 · Audio diffusion (Mel spectrograms)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHITCRAFTSYT/difflab/blob/main/notebooks/04_audio_diffusion.ipynb)

Diffuse Mel-spectrogram images, then reconstruct audio with Griffin-Lim.

In [ ]:
# === difflab bootstrap — makes `import difflab` and the configs work anywhere ===
# Works on Colab/Kaggle (no checkout) and on a local clone. Three install paths,
# tried in order: already-installed -> GitHub clone -> uploaded difflab.zip.
import glob, os, pathlib, subprocess, sys

def _have_difflab():
    try:
        import difflab  # noqa: F401
        return True
    except ModuleNotFoundError:
        return False

ROOT = None
if not _have_difflab():
    # (1) If you pushed difflab to GitHub, set DIFFLAB_REPO and it clones+installs:
    REPO_URL = os.environ.get('DIFFLAB_REPO', '')  # e.g. 'https://github.com/you/difflab.git'
    if REPO_URL:
        subprocess.run(['git', 'clone', '-q', REPO_URL, '/content/difflab'], check=False)
        ROOT = '/content/difflab'
    else:
        # (2) Else upload difflab.zip via the Files panel (left sidebar), then run this.
        hits = glob.glob('/content/difflab.zip') or glob.glob('difflab.zip')
        if not hits:
            from google.colab import files  # type: ignore
            print('Select difflab.zip to upload...')
            files.upload()
            hits = glob.glob('difflab.zip') or glob.glob('/content/difflab.zip')
        subprocess.run(['unzip', '-q', '-o', hits[0], '-d', '/content/difflab_pkg'], check=False)
        found = glob.glob('/content/difflab_pkg/**/pyproject.toml', recursive=True)
        ROOT = str(pathlib.Path(found[0]).parent) if found else '/content/difflab_pkg'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', ROOT], check=False)

# Resolve the repo root so we can find configs/ whether installed remotely or locally.
if ROOT is None:
    here = pathlib.Path.cwd()
    ROOT = str(here.parent if here.name == 'notebooks' else here)
CONFIGS = str(pathlib.Path(ROOT) / 'configs')

import difflab, torch
print('difflab', difflab.__version__, '| configs:', CONFIGS)
print('CUDA available:', torch.cuda.is_available())
# Audio extras:
# !pip install -q librosa soundfile

## Inspect the audio↔mel bridge (CPU, synthetic sine)

In [ ]:
import numpy as np
from difflab.data.audio import MelConverter

conv = MelConverter(sample_rate=22050, n_fft=1024, hop_length=256, n_mels=64, sample_size=64)
t = np.linspace(0, conv.slice_samples/conv.sample_rate, conv.slice_samples, endpoint=False)
wav = 0.5*np.sin(2*np.pi*440*t).astype('float32')
img = conv.audio_to_image(wav)
print('mel image:', tuple(img.shape), 'range', float(img.min()), float(img.max()))
recon = conv.image_to_audio(img, n_iter=32)
print('reconstructed waveform samples:', recon.shape)

> **Compute note.** The cell below runs the *smoke* config — a tiny model and 2 optimizer steps — so it finishes in seconds on CPU and proves the pipeline. For real results, switch to the full config on a GPU runtime (Runtime → Change runtime type → GPU).

## Train the audio diffusion model

In [ ]:
from difflab.config import load_config
from difflab.training import audio

cfg = load_config(f'{CONFIGS}/audio_diffusion_smoke.yaml')  # -> audio_diffusion.yaml on GPU
result = audio.run(cfg)
result

## Sample a spectrogram and turn it into audio

In [ ]:
import torch
from diffusers import UNet2DModel
from difflab.models import build_scheduler
from difflab.sampling import ddim_sample
from difflab.data.audio import make_mel_converter
from difflab.utils import get_device

device = get_device()
model = UNet2DModel.from_pretrained(result.checkpoint_dir).to(device)
sched = build_scheduler(cfg.scheduler, 'ddim')
mel = ddim_sample(model, sched, 1, num_inference_steps=10, device=device)
conv = make_mel_converter(cfg.data, cfg.model.sample_size)
wav = conv.image_to_audio(mel[0], n_iter=16)
print('generated waveform samples:', wav.shape)
# from IPython.display import Audio; Audio(wav, rate=cfg.data.sample_rate)